# 📌 Topic 4: PyTorch Fundamentals & Building MLPs
> **Deep Learning Crash Course — Part 4**

In this notebook, we move from NumPy to **PyTorch**, the standard framework used by AI researchers and engineers worldwide!

We will cover:
1. **PyTorch Tensors** and automatic differentiation (`torch.autograd`).
2. Building neural network layers with `torch.nn.Module`.
3. Loading datasets with `DataLoader`.
4. Writing a standard PyTorch Training & Validation Loop.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using PyTorch Device: {device}")


### 4.1 PyTorch Tensors & Autograd
PyTorch tensors are like NumPy arrays, but with **GPU support** and **Automatic Differentiation**:


In [ ]:
# Autograd Demonstration
x = torch.tensor(3.0, requires_grad=True)
y = x ** 2 + 5 * x + 2 # y = x^2 + 5x + 2

# Compute derivative dy/dx = 2x + 5
y.backward()
print(f"Computed derivative dy/dx at x=3: {x.grad.item()} (Expected: 2(3)+5 = 11)")


### 4.2 Building a PyTorch Multi-Layer Perceptron (MLP)


In [ ]:
class PyTorchMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(PyTorchMLP, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            
            nn.Linear(hidden_dim, output_dim)
        )
        
    def forward(self, x):
        return self.net(x)

# Prepare Data
X, y = make_moons(n_samples=600, noise=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

train_ds = TensorDataset(torch.FloatTensor(X_train), torch.FloatTensor(y_train).unsqueeze(1))
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)

X_val_t = torch.FloatTensor(X_val).to(device)
y_val_t = torch.FloatTensor(y_val).unsqueeze(1).to(device)

# Instantiate PyTorch Model
model = PyTorchMLP(input_dim=2, hidden_dim=32, output_dim=1).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# Training Loop
train_losses, val_losses = [], []
for epoch in range(100):
    model.train()
    b_losses = []
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        out = model(bx)
        loss = criterion(out, by)
        loss.backward()
        optimizer.step()
        b_losses.append(loss.item())
        
    train_loss = np.mean(b_losses)
    train_losses.append(train_loss)
    
    model.eval()
    with torch.no_grad():
        v_loss = criterion(model(X_val_t), y_val_t).item()
        val_losses.append(v_loss)

print(f"Final Train Loss: {train_losses[-1]:.4f} | Val Loss: {val_losses[-1]:.4f}")

plt.figure(figsize=(8, 4))
plt.plot(train_losses, label="Train Loss", color="teal", lw=2)
plt.plot(val_losses, label="Validation Loss", color="coral", lw=2)
plt.title("PyTorch MLP Training Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()
